# Deploy DeepSeek-V4.1-Flash on an Amazon SageMaker AI Endpoint (ml.g7e.48xlarge)

## DeepSeek-V4.1-Flash: Million-Token Context with a Mixture-of-Experts Model

📄 [Model card](https://huggingface.co/deepseek-ai/DeepSeek-V4.1-Flash)

### Introduction

**DeepSeek-V4.1-Flash** is a multimodal Mixture-of-Experts (MoE) language model (552B backbone parameters plus a 196B Engram memory; 8B activated per token in prefill, 16B in decode) that supports a context length of **one million tokens**. The released checkpoint is natively quantized (FP8 weights with FP4 experts, ~475 GiB), so vLLM loads it directly without a separate quantization step.

This notebook deploys the model, from weights staged in S3, on an Amazon SageMaker AI **real-time endpoint** using the **AWS vLLM Deep Learning Container** (DLC), then runs text, reasoning, and streaming inference examples.

> **Prerequisites**
> - Weights already staged to S3 (see the **One-time setup** section of the [README](README.md)).
> - An IAM role with SageMaker execution permissions and read access to the weights bucket.
> - Quota for **`ml.p6-b200.48xlarge`** (8× B200); a capacity reservation is recommended.
>
> The endpoint uses the **`al2023-ami-sagemaker-inference-gpu-4-1`** inference AMI (driver 580 / CUDA 13), which matches the DLC's CUDA-13 build.

> **This notebook targets `ml.g7e.48xlarge`** (8× NVIDIA RTX PRO 6000 Blackwell Server Edition, 96 GB each, PCIe). The stock vLLM release does not yet start DeepSeek-V4.1 on this GPU (SM120), so it uses a custom image built from [`container-g7e/`](container-g7e/) — see the README section **ml.g7e.48xlarge**. For `ml.p6-b200.48xlarge`, use [`DeepSeek-V4.1.ipynb`](DeepSeek-V4.1.ipynb).


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to interact with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to interact with SageMaker endpoints

In [ ]:
#
# Helper functions to remove the dependency on the SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves the 'Creating' or 'Updating' state."""
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## One-time setup: stage the weights and build the image

1. **Weights** — stage the ~475 GiB checkpoint to S3 once with [`buildspec-weights.yml`](buildspec-weights.yml) (same job as the B200 notebook; README → *One-time setup*). SageMaker mounts it read-only at `/opt/ml/model`.
2. **Image** — build the g7e serving image once with [`buildspec-image-g7e.yml`](buildspec-image-g7e.yml) and push it to your ECR repository (README → *ml.g7e.48xlarge*).

Set your bucket below; the cell verifies the weights are present before you deploy.


In [ ]:
bucket = "<YOUR_BUCKET>"                   # bucket you staged the weights into (see README → One-time setup)
weights_prefix = "models/deepseek-v41-flash"
weights_s3_uri = f"s3://{bucket}/{weights_prefix}/"

# Verify the weights are present before deploying.
s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=bucket, Prefix=weights_prefix, MaxKeys=1)
assert resp.get("KeyCount", 0) > 0, (
    f"No objects under s3://{bucket}/{weights_prefix}/ — "
    "run the weight-staging CodeBuild job first (see README → One-time setup)."
)
print("weights staged at:", weights_s3_uri)

## Container

The serving image is **your ECR copy of [`container-g7e/`](container-g7e/)**: the official vLLM **v0.31.0** release plus three changes needed on RTX PRO 6000 (SM120) — FlashInfer 0.7.1rc5, DeepGEMM `17ca18f7`, and a small vLLM patch that sets a 64-token KV block for the DeepSeek-V4.1 attention and indexer backends ([vllm#59203](https://github.com/vllm-project/vllm/issues/59203)). A thin nginx layer serves the SageMaker `/ping` + `/invocations` contract, and `SM_VLLM_*` environment variables map to `vllm serve` flags, the same convention as the AWS vLLM DLC.


In [ ]:
instance = {"type": "ml.g7e.48xlarge", "num_gpu": 8}

# Your ECR image built from container-g7e/ (see README -> ml.g7e.48xlarge).
account_id = boto3.client("sts").get_caller_identity()["Account"]
inference_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/dsv41-sm120:v031"

# Driver 580 / CUDA 13 — matches the image's cu130 build.
inference_ami = "al2023-ami-sagemaker-inference-gpu-4-1"

model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 3600
variant_name = "v1"
print(inference_image)
print(weights_s3_uri)


### vLLM configuration

This is the configuration validated on 8× RTX PRO 6000:
- **`SM_VLLM_BLOCK_SIZE=64` is required.** `128` (the vLLM default for this model) fails on SM120 — the indexer kernel only accepts 32- or 64-token pages.
- **`SM_VLLM_ATTENTION_CONFIG={"indexer_kv_dtype":"fp8"}` is required.** The default MXFP4 indexer cache is only supported on B200-class GPUs.
- Tensor parallelism across all 8 GPUs, an **FP8 KV cache**, and `VLLM_PLE_CPU_OFFLOAD=1` to keep the Engram memory table in host memory. Do **not** set it to `0`.
- `SM_VLLM_MAX_MODEL_LEN` starts at 65536 for a fast smoke test. The same image served `1048576` (1M) tokens on g7e.48xlarge (KV cache ≈ 15.5M tokens at TP8).
- Validated text-only (`{"image":0}`); image inputs were not tested on this GPU.


In [ ]:
common_env = {
    # Offload the Engram memory table to host memory — REQUIRED. Never set to 0.
    "VLLM_PLE_CPU_OFFLOAD": "1",
}

vllm_env = {
    "SM_VLLM_MODEL": "/opt/ml/model",  # S3-staged weights mounted here (ModelDataSource below)
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_TRUST_REMOTE_CODE": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_BLOCK_SIZE": "64",                                    # required on SM120 (see above)
    "SM_VLLM_ATTENTION_CONFIG": json.dumps({"indexer_kv_dtype": "fp8"}),  # required on SM120
    "SM_VLLM_MAX_MODEL_LEN": "65536",  # smoke test; validated up to 1048576
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.92",
    "SM_VLLM_LIMIT_MM_PER_PROMPT": json.dumps({"image": 0}),     # text-only (validated)
    "SM_VLLM_REASONING_PARSER": "deepseek_v41",
    "SM_VLLM_TOOL_CALL_PARSER": "deepseek_v41",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    "SM_VLLM_MAX_NUM_BATCHED_TOKENS": "16384",
    "SM_VLLM_ENABLE_PREFIX_CACHING": "true",
    "SM_VLLM_MAX_NUM_SEQS": "8",
}
env = common_env | vllm_env

reasoning_keyword = "reasoning"


## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
        # Mount the uncompressed S3 weights read-only at /opt/ml/model.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": weights_s3_uri,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
    },
)

In [ ]:
capacity_reservation = {
    "CapacityReservationPreference": "capacity-reservations-only",
    "MlReservationArn": "<YOUR_ML_RESERVATION_ARN_HERE>",  # replace with your reservation ARN
}

_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "ModelDataDownloadTimeoutInSeconds": timeout,
            "InferenceAmiVersion": inference_ami,
            # Uncomment if you are deploying against a capacity reservation.
            # "CapacityReservationConfig": capacity_reservation,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)

# Cold start includes the image pull, weight load, and engine warmup — this can take up to ~60 minutes.
wait_for_endpoint(endpoint_name)

## Inference examples

### Text inference

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get("content", None)
if reasoning:
    display(Markdown("### Reasoning:\n---"))
    display(Markdown(reasoning))
if content:
    display(Markdown("### Content:\n---"))
    display(Markdown(content))

### Text inference with reasoning

Thinking is on by default. Set the reasoning effort via `chat_template_kwargs`: an integer from 1 to 100, or `"low"` (25), `"high"` (50, the default), `"xhigh"` (75), or `"max"` (100). The reasoning text is returned in the `reasoning` field of the message.

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "Analyze why speculative decoding works better for long-context language models."}
    ],
    "chat_template_kwargs": {
        "thinking": True,
        "reasoning_effort": "high",  # or "low" / "xhigh" / "max", or an integer 1-100
    },
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get("content", None)
if reasoning:
    display(Markdown("### Reasoning:\n---"))
    display(Markdown(reasoning))
if content:
    display(Markdown("### Content:\n---"))
    display(Markdown(content))

### Streaming example with reasoning enabled

In [ ]:
#
# Helper for streaming invocation
#
import io

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                print("Unknown event type:" + str(chunk))
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])


def stream_response(endpoint_name, inputs):
    body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": inputs}]}
        ],
        "stream": True,
        "chat_template_kwargs": {
            "thinking": True,
            "reasoning_effort": "max",  # or "low" / "high" / "xhigh", or an integer 1-100
        },
    }

    resp = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        Body=json.dumps(body),
        ContentType="application/json",
    )

    event_stream = resp["Body"]
    start_json = b"{"
    full_response = ""
    start_time = time.time()
    reasoning_token_count = 0
    content_token_count = 0
    tps = 0

    for line in LineIterator(event_stream):
        if line != b"" and start_json in line:
            data = json.loads(line[line.find(start_json):].decode("utf-8"))
            delta = data["choices"][0]["delta"]
            reasoning = delta.get(reasoning_keyword, "")
            content = delta.get("content", "")
            token_text = ""
            if reasoning:
                if reasoning_token_count == 0:
                    token_text = "### Reasoning:\n"
                token_text += reasoning
                reasoning_token_count += 1
            if content and content != "":
                if content_token_count == 0:
                    token_text = "\n### Content:\n"
                token_text += content
                content_token_count += 1
            if token_text:
                full_response += token_text

            elapsed_time = time.time() - start_time
            tps = (reasoning_token_count + content_token_count) / elapsed_time if elapsed_time > 0 else 0

            clear_output(wait=True)
            display(Markdown(full_response))

    print(f"\nTokens per second: {tps:.2f}\n")

In [ ]:
inputs = "Solve this problem step by step: What is 15% of 240?"
stream_response(endpoint_name, inputs)

## Cleanup

Delete the endpoint and associated resources to stop charges. The staged weights (S3) persist for future deploys — remove them separately when you no longer need them.

In [ ]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)